In [1]:
%pip install jupysql duckdb duckdb-engine --quiet

%load_ext sql
%sql duckdb:///:memory:

Note: you may need to restart the kernel to use updated packages.


Connecting to 'duckdb:///:memory:'

In [2]:
%%sql
create table bookings as select * from read_csv_auto('bookings.csv');
create table clients as select * from read_csv_auto('clients.csv');

Running query in 'duckdb:///:memory:'

Count
50


In [5]:
%%sql

with bookings_with_tier as (
    select 
        booking_id,
        a.client_id as client_id, 
        subproduct,
        booking_date,
        trip_date,
        amount_rub,
        status,
        tier
    from bookings as a left join clients as b
    on a.client_id = b.client_id
),
avg_check as (
    select
        tier,
        round(avg(amount_rub),2) as avg_check,
        count(*) as bookings_count
    from bookings_with_tier
    group by tier 
),

clients_count as (
    select
        tier,
        count(*) as clients_count
    from clients 
    group by tier 
),

avg_check_clients_count as (
    select 
        a.tier as tier, 
        avg_check,
        bookings_count,
        clients_count
    from avg_check as a left join clients_count as b
    on a.tier = b.tier
)


select
    tier,
    avg_check,
    bookings_count,
    clients_count, 
    round(1.0 * bookings_count / clients_count,2) as avg_bookings_count 
from avg_check_clients_count
order by tier 

Running query in 'duckdb:///:memory:'

tier,avg_check,bookings_count,clients_count,avg_bookings_count
1,82848.48,474,4,118.5
2,44750.29,976,16,61.0
3,29451.46,514,30,17.13


In [ ]:
Гипотеза: клиенты тира 1 бронируют более дорогие продукты, но реже, чем клиенты тира 3.

Я намеренно считаю абсолютно все бронирования даже со статусом cancelled.
Так как в гипотезе речь идет именно о бронировании, пусть даже и отмененном впоследствии.

Клиенты тира 1 действительно бронируют более дорогие продукты, чем клиенты тира 3. Это видно по среднему чеку.

Клиент тира 1 в среднем сделал 118.5 бронирований. Клиент тира 3 в среднем сделал 17.13 бронирований. 
То есть клиенты тира 1 совершают в разы больше бронирований, чем клиенты тира 3. 

Гипотиза неверна. 

Верное утверждение, проверенное на данных: клиенты тира 1 бронируют более дорогие продукты и делают это чаще, чем клиенты тира 3.
    
Отдельно важно, что у тира 1 всего 4 клиента. 
Это означает, что выводы по тиру 1 особенно чувствительны к поведению каждого отдельного клиента.
Поэтому любые бизнес-решения по этому тиру нужно проверять дополнительно, так как выборка очень маленькая.